In [13]:
import os
import base64
import requests
import io
import json
from google import genai
import time as time_sleep
from elevenlabs.client import ElevenLabs
from google.cloud import bigquery, storage
from pathlib import Path
from google.oauth2 import service_account
from typing import Optional
import cv2
from PIL import Image
import subprocess
import glob
import imageio_ffmpeg

In [ ]:
PROJECT_ID="ferrer-odiseia4good-sandbox"
DATASET_ID="basedatos"

GOOGLE_CLOUD_GEMINI_API_KEY="<secret>"
GOOGLE_CLOUD_GEMINI_API_KEY_RAFA="<secret>"

ELEVENLABS_API_KEY="<secret>"
ELEVENLAB_VOICE_ID="gD1IexrzCvsXPHUuT0s3"

GCS_BUCKET_SALIDAS = "odiseia-salidas-images"
NO_VIDEOS=3
VERTEX_AI_RETIRES = 5
VERTEX_AI_SECONDS_SLEEP=10

SCRIPT_DIR = Path.cwd()
GOOGLE_CREDENTIALS = "app/credentials/credentials.json"
GOOGLE_CREDENTIALS_RAFA = "app/credentials/credentials_rafa.json"
GOOGLE_CREDENTIALS_PATH = SCRIPT_DIR / GOOGLE_CREDENTIALS
GOOGLE_CREDENTIALS_PATH_RAFA = SCRIPT_DIR / GOOGLE_CREDENTIALS_RAFA

In [3]:
def download_base64_from_gcs(filename: str, bucket_name: str) -> Optional[str]:
    try:
        bucket = storage_client.bucket(bucket_name)
        blob = bucket.blob(filename)
        if not blob.exists():
            return None
        content = blob.download_as_bytes()
        return base64.b64encode(content).decode('utf-8')
    except Exception as e:
        print(f"Error al descargar el archivo {filename} del bucket {bucket_name}: {e}")
        return None

def b64decode(b64_encoded_string: str) -> bytes:
  return base64.b64decode(b64_encoded_string.encode('utf-8'))

def extraer_ultimo_fotograma_base64(video_path):
    # Abrir el video
    cap = cv2.VideoCapture(video_path)
    
    # Obtener el número total de fotogramas
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Ir al último fotograma
    cap.set(cv2.CAP_PROP_POS_FRAMES, total_frames - 1)
    
    # Leer el último fotograma
    ret, frame = cap.read()
    cap.release()
    
    if ret:
        # Convertir de BGR (OpenCV) a RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Convertir a PIL Image
        pil_image = Image.fromarray(frame_rgb)
        
        # Guardar en buffer como PNG
        buffer = io.BytesIO()
        pil_image.save(buffer, format='PNG')
        buffer.seek(0)
        
        # Convertir a base64
        img_base64 = base64.b64encode(buffer.read()).decode('utf-8')
        
        return img_base64
    else:
        raise Exception("No se pudo leer el último fotograma del video")
    
def voice_convert(
    input_audio_path: str,
    output_audio_path: str,
    model_id: str = "eleven_multilingual_sts_v2",   # STS (speech-to-speech)
    output_format: str = "mp3_44100_128",           # o "wav"
    stability: float = 0.5,
    similarity_boost: float = 0.75,
    style: float = 0.0,
    use_speaker_boost: bool = True,
):

    voice_id = ELEVENLAB_VOICE_ID

    url = f"https://api.elevenlabs.io/v1/speech-to-speech/{voice_id}"

    headers = {
        "xi-api-key": ELEVENLABS_API_KEY,
        # Importante: NO poner Content-Type aquí; requests lo gestiona con multipart/form-data
        "accept": "audio/mpeg" if output_format.startswith("mp3") else "audio/wav",
    }

    data = {
        "model_id": model_id,
        "output_format": output_format,
        "voice_settings": json.dumps({
            "stability": stability,
            "similarity_boost": similarity_boost,
            "style": style,
            "use_speaker_boost": use_speaker_boost,
        }),
    }

    with open(input_audio_path, "rb") as f:
        files = {
            "audio": (os.path.basename(input_audio_path), f),
        }

        r = requests.post(url, headers=headers, data=data, files=files, timeout=180)
        if r.status_code != 200:
            raise RuntimeError(f"ElevenLabs error {r.status_code}: {r.text}")

        with open(output_audio_path, "wb") as out:
            out.write(r.content)

In [4]:
credentials = service_account.Credentials.from_service_account_file(str(GOOGLE_CREDENTIALS_PATH))
credentials_rafa = service_account.Credentials.from_service_account_file(str(GOOGLE_CREDENTIALS_PATH_RAFA)).with_scopes(['https://www.googleapis.com/auth/cloud-platform'])

client_genia = genai.Client(
      vertexai=True,
      api_key=GOOGLE_CLOUD_GEMINI_API_KEY,
  )

client_genia_rafa = genai.Client(
      vertexai=True,
      project="nano-banana-472908",
      location="us-central1",
      credentials=credentials_rafa
  )

client_elevenlabs = ElevenLabs(
    api_key=ELEVENLABS_API_KEY
)

storage_client = storage.Client(project=PROJECT_ID, credentials=credentials)

client_bigquery = bigquery.Client(project=PROJECT_ID, credentials=credentials)

In [15]:
ranking=2
id_alumno="odiseia_ferrer_monica"
id_centro="ferrerodiseia"

In [16]:
sql = f"""SELECT distinct id_salida FROM `{PROJECT_ID}.{DATASET_ID}.T_ALUMNOS_SALIDAS`
WHERE id_alumno=@id_alumno AND id_centro=@id_centro AND ranking=@ranking
LIMIT 1
"""
job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("id_alumno", "STRING", id_alumno),
        bigquery.ScalarQueryParameter("id_centro", "STRING", id_centro),
        bigquery.ScalarQueryParameter("ranking", "INTEGER", ranking)
    ]
)
query_job = client_bigquery.query(sql, job_config=job_config)
salidas = query_job.result()
salidas_list = list(salidas)
id_salida = salidas_list[0].id_salida if salidas_list else 0
print(id_salida)

-151980953004141950


In [17]:
sql = f"""SELECT  distinct titulo, que_voy_a_aprender FROM `{PROJECT_ID}.{DATASET_ID}.T_SALIDAS_PROFESIONALES`
WHERE id_salida = @id_salida
LIMIT 1
"""
job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("id_salida", "INTEGER", id_salida)
    ]
)
query_job = client_bigquery.query(sql, job_config=job_config)
salidas = query_job.result()

salidas_list = list(salidas)
titulo = salidas_list[0].titulo if salidas_list else None
que_voy_a_aprender = salidas_list[0].que_voy_a_aprender if salidas_list else None

In [ ]:
#Generamos el prompt:
model = "gemini-2.5-pro"

parts=[]

imagen_filename= f"{id_centro}/{id_alumno}/{titulo}.png"
imagen_filename_base64=download_base64_from_gcs(imagen_filename, GCS_BUCKET_SALIDAS)

parts.append(genai.types.Part.from_text(text=f"Eres una persona que ha estudiado el {titulo}."))
parts.append(genai.types.Part.from_text(text=f"Esta eres tú:"))
parts.append(genai.types.Image(image_bytes=b64decode(imagen_filename_base64),mime_type="image/png")),
parts.append(genai.types.Part.from_text(text=f"Imagina que estás hablando con tu 'yo del pasado' antes de decidirte a estudiar {titulo}."))
parts.append(genai.types.Part.from_text(text="Vas a mantener un discurso con tu 'yo del pasado' contándole 4 puntos:"))
parts.append(genai.types.Part.from_text(text=f"En el primer punto vas a saludar a tu ¡yo del pasado', refiriendote a él como 'yo del pasado', dile que eres su yo del futuro si hubiera estudiado el {titulo} y dile lo contento que estás por haber tomado esa decisión."))
parts.append(genai.types.Part.from_text(text=f"En el segundo punto coméntale todo lo que aprendiste en {titulo}. Para darte más contexto, esto es todo lo que aprendiste:{que_voy_a_aprender}"))             
parts.append(genai.types.Part.from_text(text=f"En el tercer punto coméntale cómo es tu día a día. Que funciones hacer y porqué te motivan."))
parts.append(genai.types.Part.from_text(text=f"En el cuarto y último punto despidete animándole a apuntarse en {titulo}."))             
parts.append(genai.types.Part.from_text(text="Importante:"))
parts.append(genai.types.Part.from_text(text="La prsona a la que le hablas es tu 'yo del pasado' y tienes la finalidad de animarle a proseguir en sus estudios y a que termine cursando el {titulo}"))
parts.append(genai.types.Part.from_text(text="Que el discurso sea motivador pero sin exagerar. Debe ayudar a un usuario a conectar con su yo del futuro."))
parts.append(genai.types.Part.from_text(text="Que cada punto esté claramente diferenciado y estructurado pero a la vez, tengan sentido secuencial, es decir, que el punto 2 se pueda leer inmediatamente después del 1 y así sucesivamente."))
parts.append(genai.types.Part.from_text(text="Cada punto debe tener una extensión aproximada de 20 palabras."))
    
properties_json={}
properties_json['parte_1']= {"type":"STRING","description":"Primer fragmento de la conversación. Debe tener una extensión aproximada de 20 palabras."}
properties_json['parte_2']= {"type":"STRING","description":"Segundo fragmento de la conversación. Debe tener una extensión aproximada de 20 palabras."}
properties_json['parte_3']= {"type":"STRING","description":"Tercer fragmento de la conversación. Debe tener una extensión aproximada de 20 palabras."}
properties_json['parte_4']= {"type":"STRING","description":"Cuarto fragmento de la conversación. Debe tener una extensión aproximada de 20 palabras."}
properties_list=['parte_1','parte_2','parte_3','parte_4']

contents = [genai.types.Content(role="user",parts=parts)]

generate_content_config = genai.types.GenerateContentConfig(
    temperature = 0,
    top_p = 1,
    seed=0,
    max_output_tokens = 32768,
    safety_settings = 
                    [ 
                        genai.types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH",threshold="OFF"),
                        genai.types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT",threshold="OFF"),
                        genai.types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT",threshold="OFF"),
                        genai.types.SafetySetting(category="HARM_CATEGORY_HARASSMENT",threshold="OFF"),
                    ],
    response_mime_type = "application/json",
    response_schema = {"type":"OBJECT","properties":properties_json,"required":properties_list}
)

response = client_genia.models.generate_content(model=model,contents=contents,config=generate_content_config)
output=response.candidates[0].content.parts[0].text

output = json.loads(output)
text_chunks=[]
text_chunks.append(output.get("parte_1"))
text_chunks.append(output.get("parte_2"))
text_chunks.append(output.get("parte_3"))
text_chunks.append(output.get("parte_4"))
for text_chunk in text_chunks:
    print(text_chunk)

Hola, yo del pasado. Soy tu yo del futuro que estudió Artes Gráficas, y no sabes lo contento que estoy.
Aprendí a gestionar encargos, tratar imágenes, archivar documentos, hacer presupuestos y a preparar todos los materiales para la producción.
Mi día a día es muy creativo, desde atender clientes a preparar diseños. Me motiva ver cómo las ideas cobran vida.
No lo dudes más y anímate a apuntarte. Te espera un futuro apasionante en el mundo de las Artes Gráficas.


In [32]:
model="veo-3.1-generate-001"

videos_generados=[]

ffmpeg_exe = imageio_ffmpeg.get_ffmpeg_exe()

for i, text_chunk in enumerate(text_chunks):

    prompt = f"El usuario mira a cámara sonriendo y diciendo lo siguiente: {text_chunk}. Lo dice en un perfecto castellano con acento peninsular."
    
    source = genai.types.GenerateVideosSource(
        prompt=prompt,
        image=genai.types.Image(
            image_bytes=b64decode(imagen_filename_base64),
            mime_type="image/png",
        ),
    )

    config = genai.types.GenerateVideosConfig(
        aspect_ratio="16:9",
        number_of_videos=1,
        duration_seconds=8,
        person_generation="allow_all",
        generate_audio=True,
        resolution="720p",
        seed=0,
        last_frame=genai.types.Image(
            image_bytes=b64decode(imagen_filename_base64),
            mime_type="image/png",
        ),
    )

    operation = client_genia_rafa.models.generate_videos(
        model=model, source=source, config=config
    )

    # Waiting for the video(s) to be generated
    while not operation.done:
        print("Video has not been generated yet. Check again in 10 seconds...")
        time_sleep.sleep(10)
        operation = client_genia_rafa.operations.get(operation)

    # Acceder a los videos generados desde el resultado de la operación
    result = operation.result
    generated_videos = result.generated_videos if hasattr(result, 'generated_videos') else [result]

    for generated_video in generated_videos:
        if generated_video.video:
            video_filename = f"veo_video_{i}.mp4"
            with open(video_filename, 'wb') as f:
                f.write(generated_video.video.video_bytes)
            print(f"Video guardado como: {video_filename}")

    imagen_filename_base64 = extraer_ultimo_fotograma_base64(video_filename)

    audio_file_name = f"veo_audio_{i}.mp3"
    video_filename_con_audio = f"veo_video_{i}_con_audio.mp4"
    audio_file_name_transcripted = f"veo_audio_{i}_transcripted.mp3"
        
    cmd = [
        ffmpeg_exe,
        '-i', video_filename,
        '-vn',
        '-acodec', 'mp3',
        '-ab', '192k',
        '-y',
        audio_file_name
    ]
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode == 0:
        print(f"  ✓ Audio extraído: {audio_file_name}")
    else:
        print(f"  ✗ Error: {result.stderr}")

    voice_convert(
        input_audio_path=audio_file_name,
        output_audio_path=audio_file_name_transcripted,
        output_format="mp3_44100_128",
        stability=0.45,
        similarity_boost=0.8,
        style=0.1,
        use_speaker_boost=True,
    )


    cmd = [
        ffmpeg_exe,
        '-i', video_filename,      # Video sin audio
        '-i', audio_file_name_transcripted,       # Audio
        '-c:v', 'copy',          # Copiar video sin recodificar
        '-c:a', 'aac',           # Codificar audio a AAC
        '-map', '0:v:0',         # Usar video del primer archivo
        '-map', '1:a:0',         # Usar audio del segundo archivo
        '-shortest',              # Usar la duración más corta
        '-y',                     # Sobrescribir sin preguntar
        video_filename_con_audio
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode == 0:
        print(f"✓ Video combinado guardado como: {video_filename_con_audio}")
    else:
        print(f"✗ Error: {result.stderr}")

    videos_generados.append(video_filename_con_audio)
    

Video has not been generated yet. Check again in 10 seconds...
Video has not been generated yet. Check again in 10 seconds...
Video has not been generated yet. Check again in 10 seconds...
Video has not been generated yet. Check again in 10 seconds...
Video has not been generated yet. Check again in 10 seconds...
Video has not been generated yet. Check again in 10 seconds...
Video has not been generated yet. Check again in 10 seconds...
Video guardado como: veo_video_0.mp4
  ✓ Audio extraído: veo_audio_0.mp3
✓ Video combinado guardado como: veo_video_0_con_audio.mp4
Video has not been generated yet. Check again in 10 seconds...
Video has not been generated yet. Check again in 10 seconds...
Video has not been generated yet. Check again in 10 seconds...
Video has not been generated yet. Check again in 10 seconds...
Video has not been generated yet. Check again in 10 seconds...
Video has not been generated yet. Check again in 10 seconds...
Video has not been generated yet. Check again in 